# Análisis de Cobertura Indoor en un Edificio Real

Este notebook desarrolla un ejercicio de penetración indoor para un edificio de varias plantas, evaluando si la cobertura exterior es suficiente para voz, mensajería y acceso a plataformas digitales, o si es necesario un refuerzo indoor (DAS, repetidor o small cell).

**Puntos cubiertos:**
- Pérdidas outdoor-to-indoor (fábrica, paredes, forjados).
- Presupuesto de enlace y niveles recibidos.
- Comparación con umbrales de servicio (voz y datos básicos).
- Propuesta de mejora y análisis de frecuencias.


## 1. Importar librerías necesarias

Importamos las librerías necesarias para los cálculos y la visualización. Si se quiere integrar con QGIS se puede usar `geopandas` y `rasterio`, pero aquí trabajaremos con herramientas estándar.


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ajuste de estilo para gráficos
plt.style.use('seaborn-whitegrid')


## 2. Definir parámetros del escenario y puntos de usuario

Definimos los parámetros de portadora (LTE 1800 o UMTS 2100), la antena exterior, y las pérdidas típicas de fachada, paredes interiores y forjados. Además se definen los tres puntos de usuario: entrada, aula segunda planta y semisótano.


In [ ]:
# Parámetros de la señal (potencia de transmisión y ganancia de antena)
Pt_dBm = 43  # Potencia isotrópica equivalente (dBm) - ejemplo 43 dBm (20 W)
Gt_dBi = 14  # Ganancia de antena exterior (dBi)
Gr_dBi = 0   # Ganancia de antena de usuario (dBi) - típico de terminal móvil

# Altura de antena y distancias exteriores
h_antena_m = 22  # altura aproximada de la antena exterior (20-25 m)

# Umbrales de servicio (aproximados)
threshold_voice_dBm = -100  # voz (por ejemplo LTE RSRP o UMTS RSCP mínimo)
threshold_data_dBm = -95    # datos básicos (puede variar según velocidad requerida)

# Pérdidas típicas (valores recomendados)
loss_facade_dB = 15       # fachada (12-18 dB)
loss_wall_dB = 4          # pared interior (3-5 dB)
loss_floor_dB = 15        # forjado (12-18 dB)

# Definición de puntos de usuario con distancias aproximadas desde la fachada
# (esquema simple: la antena exterior se asume en el lado de la calle frente a la fachada)
users = {
    'Entrada': {
        'distancia_exterior_m': 30,  # distancia desde la antena exterior hasta la fachada
        'profundidad_interior_m': 2,  # distancia dentro del edificio (pasillo de entrada)
        'num_paredes': 1,
        'num_forjados': 0,
    },
    'Aula 2ª planta': {
        'distancia_exterior_m': 40,  # mayor distancia exterior, mayor recorrido
        'profundidad_interior_m': 8,  # dentro del edificio (pasillos + aula)
        'num_paredes': 2,
        'num_forjados': 2,  # penetrar forjados para subir al 2º planta
    },
    'Semisótano (archivo)': {
        'distancia_exterior_m': 35,
        'profundidad_interior_m': 12,  # recorrido interior hasta el archivo
        'num_paredes': 3,
        'num_forjados': 2,  # forjados hacia el semisótano
    },
}


## 3. Calcular pérdidas de propagación exterior

Calculamos la pérdida de propagación externa desde la antena exterior hasta la fachada del edificio. Se puede usar el modelo de pérdida en espacio libre (FSPL) o un modelo de propagación urbana simplificado. Aquí se usa FSPL como referencia y se muestra cómo ajustar con un factor de atenuación adicional.


In [ ]:
def fspl(freq_mhz: float, dist_m: float) -> float:
    """Free-space path loss (dB)"""
    if dist_m <= 0:
        return 0.0
    # FSPL = 20*log10(d) + 20*log10(f) + 32.44 (d in km, f in MHz)
    dist_km = dist_m / 1000.0
    return 20 * math.log10(dist_km) + 20 * math.log10(freq_mhz) + 32.44

def outdoor_path_loss(freq_mhz: float, dist_m: float, n: float = 3.5) -> float:
    """Modelo de pérdida en espacio libre + factor de trayectoria (log-distance)."""
    # n=2 es espacio libre, n=3-4 típico en entorno urbano denso
    if dist_m <= 0:
        return 0.0
    # Usar FSPL a 1 m como referencia
    pl_1m = fspl(freq_mhz, 1.0)
    pl = pl_1m + 10 * n * math.log10(dist_m / 1.0)
    return pl

# Ejemplo de cálculo para LTE 1800 y UMTS 2100
for f in [1800, 2100]:
    pl = outdoor_path_loss(f, 30)
    print(f'Pérdida exterior aproximada a {f} MHz y 30 m: {pl:.1f} dB')


## 4. Calcular pérdidas de penetración por fachada, paredes y forjados

Sumamos las pérdidas de penetración en función de si el punto de usuario se encuentra cerca de la fachada, dentro del edificio o en semisótano.


In [ ]:
def penetration_loss(point: dict, loss_facade: float, loss_wall: float, loss_floor: float) -> dict:
    """Calcula las pérdidas interiores para un punto de usuario."""
    losses = {}
    # Pérdida de fachada (una única vez para entrar al edificio)
    losses['fachada_dB'] = loss_facade
    # Pérdida por paredes interiores
    losses['paredes_dB'] = point['num_paredes'] * loss_wall
    # Pérdida por forjados
    losses['forjados_dB'] = point['num_forjados'] * loss_floor
    # Pérdida adicional por profundidad interior (se puede modelar como pérdida extra por distancia)
    losses['profundidad_dB'] = (point['profundidad_interior_m'] / 10.0) * 2.0  # 2 dB por 10 m dentro
    losses['total_interior_dB'] = sum(losses.values())
    return losses

# Calcular pérdidas interiores para cada punto de usuario
for name, point in users.items():
    users[name]['pierdas_interior'] = penetration_loss(point, loss_facade_dB, loss_wall_dB, loss_floor_dB)

# Mostrar resultados parciales
pd.DataFrame({k: v['pierdas_interior'] for k, v in users.items()})


## 5. Sumar pérdidas totales por tramo y crear tabla de resultados

Creamos una tabla que incluye la pérdida exterior, la pérdida de fachada y las pérdidas interiores adicionales para cada punto de usuario.


In [ ]:
def compute_link_budget(freq_mhz: float, user: dict) -> dict:
    # Pérdida exterior desde la antena hasta la fachada
    pl_exterior = outdoor_path_loss(freq_mhz, user['distancia_exterior_m'])
    # Pérdidas interiores (fachada + paredes + forjados + profundidad)
    interior = user['pierdas_interior']
    total_loss = pl_exterior + interior['total_interior_dB']
    # Nivel recibido (dBm)
    prx_dBm = Pt_dBm + Gt_dBi + Gr_dBi - total_loss
    return {
        'pl_exterior_dB': pl_exterior,
        'pl_fachada_dB': interior['fachada_dB'],
        'pl_paredes_dB': interior['paredes_dB'],
        'pl_forjados_dB': interior['forjados_dB'],
        'pl_profundidad_dB': interior['profundidad_dB'],
        'pl_total_dB': total_loss,
        'prx_dBm': prx_dBm,
    }

# Construir tabla de resultados para cada frecuencia solicitada
freqs = [1800, 2100]
results = []
for freq in freqs:
    for name, user in users.items():
        lb = compute_link_budget(freq, user)
        lb.update({
            'punto': name,
            'frecuencia_MHz': freq,
        })
        results.append(lb)

df_results = pd.DataFrame(results)
df_results[['punto', 'frecuencia_MHz', 'prx_dBm', 'pl_total_dB']]


## 6. Verificar nivel recibido frente a umbrales de servicio

Comparamos el nivel recibido en cada punto con los umbrales para voz y datos básicos y decidimos si la cobertura es suficiente.


In [ ]:
def check_service(prx_dBm: float, voice_threshold: float, data_threshold: float) -> dict:
    return {
        'voice_ok': prx_dBm >= voice_threshold,
        'data_ok': prx_dBm >= data_threshold,
        'delta_voice_dB': prx_dBm - voice_threshold,
        'delta_data_dB': prx_dBm - data_threshold,
    }

checks = []
for _, row in df_results.iterrows():
    status = check_service(row['prx_dBm'], threshold_voice_dBm, threshold_data_dBm)
    checks.append({
        'punto': row['punto'],
        'frecuencia_MHz': row['frecuencia_MHz'],
        'prx_dBm': row['prx_dBm'],
        'voice_ok': status['voice_ok'],
        'data_ok': status['data_ok'],
        'delta_voice_dB': status['delta_voice_dB'],
        'delta_data_dB': status['delta_data_dB'],
    })

df_checks = pd.DataFrame(checks)
df_checks


## 7. Proponer soluciones de mejora (DAS, repetidor, small cell)

Si alguno de los puntos no cumple los umbrales, se propone una solución de refuerzo indoor. Los enfoques comunes son: 
- Instalación de un DAS (Distributed Antenna System) para distribuir señal interna con antenas pasivas/activas.
- Repetidor (booster) que recibe señal exterior y la amplifica dentro del edificio.
- Small cell dedicada (pico/femto) que proporciona cobertura local con backhaul o conexión a la red móvil.

En el análisis se puede iterar ajustando la ganancia de la antena indoor, la potencia de transmisión de la small cell, o la ubicación de la antena exterior para cumplir los umbrales.


## 8. Visualizar croquis del edificio y puntos de medida con QGIS

Aquí se crea un croquis simple usando `matplotlib`. En un entorno real se puede cargar un plano desde QGIS (shapefile/GeoPackage) y ubicar los puntos con coordenadas reales.


In [ ]:
# Croquis simple del edificio (planta básica)
fig, ax = plt.subplots(figsize=(6, 6))
# Dibujar un rectángulo para el edificio
building = plt.Rectangle((0, 0), 20, 15, fill=False, edgecolor='black', linewidth=2)
ax.add_patch(building)

# Ubicación aproximada de puntos de usuario (coordenadas arbitrarias para el croquis)
coords = {
    'Entrada': (2, 7),
    'Aula 2ª planta': (15, 12),
    'Semisótano': (10, 3),
}

for name, (x, y) in coords.items():
    ax.scatter(x, y, s=80, c='red')
    ax.text(x + 0.3, y + 0.3, name, fontsize=10)

ax.set_xlim(-1, 21)
ax.set_ylim(-1, 16)
ax.set_aspect('equal')
ax.set_title('Croquis simplificado del edificio y puntos de medida')
ax.set_xlabel('m (planta)')
ax.set_ylabel('m (planta)')
plt.show()


## 9. Comparar el efecto de cambiar la frecuencia de portadora

Aquí comparamos las pérdidas y los niveles recibidos si se usa una portadora más baja (700/800 MHz) o más alta (2600 MHz).


In [ ]:
freqs_ext = [700, 800, 1800, 2100, 2600]
results_freq = []
for freq in freqs_ext:
    for name, user in users.items():
        lb = compute_link_budget(freq, user)
        lb.update({
            'punto': name,
            'frecuencia_MHz': freq,
        })
        results_freq.append(lb)

df_freq = pd.DataFrame(results_freq)
df_freq.head(9)

# Graficar prx por frecuencia para cada punto
fig, ax = plt.subplots(figsize=(8, 4))
for name, group in df_freq.groupby('punto'):
    ax.plot(group['frecuencia_MHz'], group['prx_dBm'], marker='o', label=name)

ax.set_xlabel('Frecuencia (MHz)')
ax.set_ylabel('Nivel recibido (dBm)')
ax.set_title('Comparación de nivel recibido según frecuencia de portadora')
ax.legend()
ax.grid(True)
plt.show()
